#### By: Peyman Shahidi
#### Created: Apr 26, 2026
#### Last Edit: Apr 26, 2026

<br>

Score **directional handoff costs** between tasks within each O\*NET occupation by querying a language model (via EDSL). For every ordered task pair $(A, B)$ with $A \ne B$, the model returns a JSON object with three keys:

- `"reasoning"` — a few sentences explaining the assessment (forced into the JSON so it can't be skipped).
- `"handoff_score"` — integer 0–10 (0 = seamless, 10 = highly disruptive).
- `"handoff_pct_of_task_time"` — non-negative number expressing handoff time as a percentage of Task A's duration.

**Output layout** under `data/computed_objects/taskHandoffCost_LLM{_sample}/`:

```
raw_outputs/                          ← occupations that are FULLY processed
  {Occupation_Title}/
    {TaskID_A}_to_{TaskID_B}.json     ← one file per directional pair
raw_outputs_in_progress/              ← partially processed occupations (staging)
  {Occupation_Title}/
    {TaskID_A}_to_{TaskID_B}.json
parsed_outputs/                       ← stage-2 concatenated CSVs
  {Occupation_Title}.csv
```

**Robustness for multi-day runs.** The pipeline is designed to survive crashes and resume cleanly:

- **Occupation-level skip.** Before doing anything, the main loop scans `raw_outputs/` and skips any occupation whose folder already exists — those are fully complete.
- **Pair-level resume.** Within an in-progress occupation, per-pair JSON files in `raw_outputs_in_progress/{occ}/` act as completion markers; on resume, only pairs without a `.json` file are sent to the LLM.
- **Atomic promotion.** When (and only when) every pair in an occupation has a successful `.json` file, the staging directory is atomically renamed to `raw_outputs/{occ}/`. This means `raw_outputs/` is always a coherent set of fully-done occupations; partially-done work lives in the staging directory until it's complete.
- **Failed parses** stay as `{pair}.failed.txt` in staging and prevent promotion until they're retried successfully on a later run.

The pipeline runs in two stages:

- **Stage 1 — LLM call.** For each non-skipped occupation, run the survey only on pending pairs.
- **Stage 2 — Post-processing.** For each completed occupation under `raw_outputs/`, concatenate per-pair JSONs into a single CSV in `parsed_outputs/`.

In [19]:
#Python
import getpass
import os
import json
import numpy as np
import pandas as pd
from collections import defaultdict
import itertools
import random

## formatting number to appear comma separated and with two digits after decimal: e.g, 1000 shown as 1,000.00
pd.set_option('float_format', "{:,.2f}".format)

import matplotlib.pyplot as plt
#%matplotlib inline
#from matplotlib.legend import Legend

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 200)

In [20]:
import subprocess

# Install caffeinate package
%pip install caffeinate

# Use macOS built-in caffeinate command for reliability
# This prevents the system from sleeping while the process is running
try:
    # Start caffeinate in the background
    caff_process = subprocess.Popen(['caffeinate', '-d'],
                                   stdout=subprocess.DEVNULL,
                                   stderr=subprocess.DEVNULL)
    print(f"Caffeinate mode ON ☕ – Device will stay awake (PID: {caff_process.pid})")
    print("System sleep is disabled while this process runs.")

    # Store the process ID for later cleanup
    caff_pid = caff_process.pid

except Exception as e:
    print(f"⚠️ Could not start caffeinate: {e}")
    print("Continuing without caffeinate - system may sleep during long processes.")
    caff_process = None
    caff_pid = None

Note: you may need to restart the kernel to use updated packages.
Caffeinate mode ON ☕ – Device will stay awake (PID: 12864)
System sleep is disabled while this process runs.


In [ ]:
main_folder_path = ".."
input_data_path = f"{main_folder_path}/data"

# Sample mode: set to a positive int to run on only that many occupations
# (writes to a separate `_sample` folder so the production output stays clean).
# In sample mode, pick the `sample_size` occupations with the FEWEST tasks
# (filtered by `max_tasks_for_sample`), so smoke tests run quickly.
# Set sample_size to None to process all occupations.
sample_size = None
max_tasks_for_sample = 10  # only used when sample_size is set

output_subfolder = 'taskHandoffCost_LLM_sample' if sample_size else 'taskHandoffCost_LLM'
output_data_path = f'{input_data_path}/computed_objects/{output_subfolder}'

# Stage 1 outputs.
# Per-pair JSON files accumulate in `in_progress_path/{occ}/` during processing;
# once every pair for an occupation is complete, the folder is atomically renamed
# into `raw_output_path/{occ}/`. Folder existence under raw_output_path therefore
# means "this occupation is fully done" — used to skip occupations on resume.
raw_output_path = f'{output_data_path}/raw_outputs'
in_progress_path = f'{output_data_path}/raw_outputs_in_progress'

# Stage 2 outputs: per-occupation concatenated CSVs.
parsed_output_path = f'{output_data_path}/parsed_outputs'

output_plot_path = f"{main_folder_path}/writeup/plots/{output_subfolder}"

In [ ]:
# Create directories if they don't exist
for path in [raw_output_path, in_progress_path, parsed_output_path, output_plot_path]:
    if not os.path.exists(path):
        os.makedirs(path)

# 1) Stage 1 — EDSL Setup and LLM Call

For each occupation we:
1. **Skip** if the final folder `raw_outputs/{Occupation_Title}/` already exists (occupation is fully done).
2. Build all $n(n-1)$ directional task pairs.
3. Filter out pairs already written to staging at `raw_outputs_in_progress/{Occupation_Title}/{pair}.json` (pair-level resume).
4. Run the EDSL survey only on the pending pairs and write per-pair JSONs to staging.
5. **Promote** the staging folder to `raw_outputs/{Occupation_Title}/` via atomic rename — but only when every pair has been written successfully.

The model is asked to return a single JSON with three keys (`reasoning`, `handoff_score`, `handoff_pct_of_task_time`); reasoning is forced into the JSON so it can't be silently dropped.

If a response fails to parse, the raw output is saved to `{pair}.failed.txt` in staging. The occupation will not be promoted while any failed pair exists; on the next run, the failed pairs are retried (the resume check looks for `.json`, not `.failed.txt`).

In [ ]:
import re
import ast
from edsl import QuestionFreeText, Scenario, Model
from textwrap import dedent


def _extract_json(s):
    """
    Extract a dict from a model response. Tries, in order:
      1. json.loads on the cleaned string (strict JSON).
      2. ast.literal_eval on the cleaned string (handles Python-dict syntax: single-quoted keys/values).
      3. Last balanced {...} block in the string, parsed by either of the above.
    Returns None if nothing yields a dict.
    """
    if not isinstance(s, str):
        return None
    cleaned = s.replace('```json', '').replace('```', '').strip()

    try:
        d = json.loads(cleaned)
        if isinstance(d, dict):
            return d
    except (json.JSONDecodeError, AttributeError, TypeError):
        pass

    try:
        d = ast.literal_eval(cleaned)
        if isinstance(d, dict):
            return d
    except (ValueError, SyntaxError):
        pass

    matches = re.findall(r'\{[^{}]*\}', cleaned, flags=re.DOTALL)
    for candidate in reversed(matches):
        try:
            d = json.loads(candidate)
            if isinstance(d, dict):
                return d
        except json.JSONDecodeError:
            pass
        try:
            d = ast.literal_eval(candidate)
            if isinstance(d, dict):
                return d
        except (ValueError, SyntaxError):
            pass
    return None


def score_occupation_handoffs(occ_code, occ_title, tasks_df,
                              raw_output_path, in_progress_path):
    """
    Score every directional pair (A, B), A != B, for one occupation.

    Robustness:
      - Per-pair JSONs accumulate in `in_progress_path/{occ}/` during the run.
      - Once EVERY pair has a successful .json file, the staging folder is
        atomically renamed into `raw_output_path/{occ}/` — that rename is the
        single completion marker for the occupation.
      - Pair-level resume: pairs already written to staging are skipped.
      - Failed parses go to .failed.txt and prevent promotion until resolved.
    """
    safe_title = occ_title.replace(" ", "_").replace("/", "_")
    final_dir = os.path.join(raw_output_path, safe_title)
    staging_dir = os.path.join(in_progress_path, safe_title)

    # Defensive: if the final folder exists, this occupation is done; skip.
    # (The main loop also checks this; this is a second line of defense.)
    if os.path.isdir(final_dir):
        return

    if tasks_df.empty or len(tasks_df) < 2:
        print(f"   ⚠️  Fewer than 2 tasks for '{occ_title}' - skipping")
        return

    os.makedirs(staging_dir, exist_ok=True)

    # All directional pairs
    tasks_records = tasks_df[['Task ID', 'Task Title']].to_dict('records')
    all_pairs = [
        {'task_a_id': a['Task ID'], 'task_a_title': a['Task Title'],
         'task_b_id': b['Task ID'], 'task_b_title': b['Task Title']}
        for a in tasks_records for b in tasks_records
        if a['Task ID'] != b['Task ID']
    ]

    # Pair-level resume: a pair is done iff its .json exists in staging
    pending = []
    for p in all_pairs:
        pair_name = f"{p['task_a_id']}_to_{p['task_b_id']}"
        out_path = os.path.join(staging_dir, f"{pair_name}.json")
        if not os.path.exists(out_path):
            pending.append((p, pair_name, out_path))

    n_done = len(all_pairs) - len(pending)
    print(f"   • {len(tasks_df)} tasks → {len(all_pairs)} total pairs (already done in staging: {n_done}, pending: {len(pending)})")

    if pending:
        scenarios = [
            Scenario({
                'occupation': occ_title,
                'task_a_title': p['task_a_title'],
                'task_b_title': p['task_b_title'],
            })
            for (p, _, _) in pending
        ]

        q_handoff = QuestionFreeText(
            question_name="handoff_assessment",
            question_text=dedent("""\
                You are an expert on workflow design for the occupation: {{ occupation }}.

                A worker has just finished this task in their workflow:
                  Task A: {{ task_a_title }}

                Their work must now be handed off to a coworker, who will take over starting with this task:
                  Task B: {{ task_b_title }}

                Assess this directional handoff (A → B) on two measures.

                (1) Disruption score (integer 0–10):
                    How disruptive is the handoff? Consider how much accumulated context, partially-completed work, or implicit state must be transferred from the first worker to the second so that the second can pick up where the first left off.
                      - 0  = seamless handoff: no meaningful context transfer needed
                      - 10 = highly disruptive handoff: extensive context and state transfer required

                (2) Handoff time as a percentage of Task A's duration (non-negative number):
                    Estimate the *additional time* the handoff itself adds, over and above the time spent doing Task A, expressed as a percentage of the time normally spent on Task A.
                      - 0    = handoff adds no time
                      - 25   = handoff adds ~25% of Task A's duration (a brief debrief, document, or status update)
                      - 100  = handoff time equals Task A's duration (e.g., walking through the entire process step by step)
                      - 200+ = handoff dominates the work

                Note that the handoff is directional: handing off from A to B may have a different cost than handing off from B to A.

                Return your answer as a single JSON object with EXACTLY these three keys:
                  - "reasoning": a few sentences (string) explaining your assessment of both measures
                  - "handoff_score": integer 0-10
                  - "handoff_pct_of_task_time": non-negative number (no % sign)

                STRICT JSON FORMATTING RULES — your output MUST be parseable by Python's json.loads:
                  - Use STRAIGHT double quotes (") for ALL keys and string values. Do NOT use single quotes ('), curly quotes (" "), or any non-ASCII quotes.
                  - Inside the "reasoning" string, do NOT use double quotes to quote phrases. If you need to quote something, use single quotes (apostrophes) instead — e.g., write the phrase 'cleaning' rather than the phrase "cleaning". Apostrophes inside words (Task A's, worker's) are fine and require no escaping.
                  - Use straight ASCII apostrophes (U+0027), not curly apostrophes (' ’).
                  - Do not include trailing commas, comments, or any text before or after the JSON object.

                Example of the EXACT format expected:
                {"reasoning": "These tasks share little intermediate state because the equipment ends in a stable, self-contained condition; the next worker only needs a brief status check.", "handoff_score": 4, "handoff_pct_of_task_time": 20}

                Return ONLY the JSON object — nothing before or after it.
            """),
        )

        try:
            model = Model("gpt-5-mini", service_name="openai_v2", temperature=0.0, max_tokens=2000)
            results = q_handoff.by(model).by(scenarios).run(progress_bar=True)
            results_df = results.to_pandas()
        except Exception as e:
            print(f"   ❌ Error running survey: {e}")
            return

        n_failed = 0
        n_written = 0
        for (p, pair_name, out_path), raw in zip(pending, results_df['answer.handoff_assessment'].values):
            d = _extract_json(raw)

            if d is None or 'handoff_score' not in d:
                failed_path = os.path.join(staging_dir, f"{pair_name}.failed.txt")
                with open(failed_path, 'w') as f:
                    f.write(raw if isinstance(raw, str) else '(empty response)')
                n_failed += 1
                continue

            pair_record = {
                "occupation_title": occ_title,
                "soc_code": occ_code,
                "task_a": {"id": int(p['task_a_id']), "title": p['task_a_title']},
                "task_b": {"id": int(p['task_b_id']), "title": p['task_b_title']},
                "reasoning": d.get('reasoning', '') or '',
                "handoff_score": d.get('handoff_score'),
                "handoff_pct_of_task_time": d.get('handoff_pct_of_task_time'),
            }
            with open(out_path, 'w') as f:
                json.dump(pair_record, f, indent=2)
            n_written += 1

        if n_failed:
            print(f"   ⚠️  {n_failed}/{len(pending)} responses failed to parse JSON (saved as .failed.txt; will retry next run)")
        print(f"   ✅ Wrote {n_written} pair files for '{occ_title}' (staging)")

    # Promotion check: every pair must have a .json file in staging
    n_complete = sum(
        1 for p in all_pairs
        if os.path.exists(os.path.join(staging_dir, f"{p['task_a_id']}_to_{p['task_b_id']}.json"))
    )
    if n_complete == len(all_pairs):
        # Atomic rename within the same filesystem — this is the single completion marker.
        os.makedirs(raw_output_path, exist_ok=True)
        os.rename(staging_dir, final_dir)
        print(f"   🎉 Occupation complete; promoted to {final_dir}")
    else:
        n_missing = len(all_pairs) - n_complete
        print(f"   ⏸  Occupation NOT promoted: {n_missing}/{len(all_pairs)} pair(s) still missing or failed (will resume next run)")

# 2) Read O*NET Dataset and Run Sample

In [24]:
# Read O*NET data
ONET = pd.read_csv(f'{input_data_path}/computed_objects/ONET_cleaned_tasks.csv')

# Drop DWA columns and dedup so each (occupation, task) appears once
ONET = ONET.drop(columns=['DWA ID', 'DWA Title']).drop_duplicates().reset_index(drop=True)
ONET = ONET[['O*NET-SOC Code', 'Occupation Title', 'Task ID', 'Task Title']].reset_index(drop=True)

# In sample mode, pick the `sample_size` occupations with the FEWEST tasks
# (filtered to those with < `max_tasks_for_sample` tasks).
if sample_size:
    task_counts = ONET.groupby(['O*NET-SOC Code', 'Occupation Title']).size().sort_values(kind='stable')
    eligible = task_counts[task_counts < max_tasks_for_sample]
    if len(eligible) < sample_size:
        print(f"⚠️  Only {len(eligible)} occupations have <{max_tasks_for_sample} tasks; taking all of them")
        sampled = eligible
    else:
        sampled = eligible.head(sample_size)
    sampled_codes = [code for (code, _title) in sampled.index]
    ONET = ONET[ONET['O*NET-SOC Code'].isin(sampled_codes)].reset_index(drop=True)
    print(f"SAMPLE MODE: {len(sampled_codes)} occupations with <{max_tasks_for_sample} tasks selected")
    for (code, title), n in sampled.items():
        print(f"   {code}  {title}  ({n} tasks → {n*(n-1)} directional pairs)")

print(f"\nNumber of unique ONET occupations in this run: {ONET['O*NET-SOC Code'].nunique():,}")
print(f"Number of unique ONET tasks in this run: {ONET['Task ID'].nunique():,}")
ONET.head()

SAMPLE MODE: 2 occupations with <10 tasks selected
   39-5093.00  Shampooers  (4 tasks → 12 directional pairs)
   47-2072.00  Pile Driver Operators  (5 tasks → 20 directional pairs)

Number of unique ONET occupations in this run: 2
Number of unique ONET tasks in this run: 9


,O*NET-SOC Code,Occupation Title,Task ID,Task Title
0,39-5093.00,Shampooers,14818,"Massage, shampoo, and condition patron's hair ..."
1,39-5093.00,Shampooers,14819,Advise patrons with chronic or potentially con...
2,39-5093.00,Shampooers,14820,"Treat scalp conditions and hair loss, using sp..."
3,39-5093.00,Shampooers,14821,Maintain treatment records.
4,47-2072.00,Pile Driver Operators,14901,Move hand and foot levers of hoisting equipmen...


In [ ]:
# Pre-flight: scan `raw_outputs/` once to find occupations that are already fully done.
# Folder existence under raw_output_path is the single completion marker (set by the
# atomic rename inside the function). We skip those occupations entirely and don't
# even build their task lists or call the function.
already_complete = set()
if os.path.isdir(raw_output_path):
    for entry in os.listdir(raw_output_path):
        if os.path.isdir(os.path.join(raw_output_path, entry)):
            already_complete.add(entry)

occupation_groups = list(ONET.groupby(['O*NET-SOC Code', 'Occupation Title'], sort=False))
n_total = len(occupation_groups)
n_skip = sum(
    1 for (_, occ_title), _ in occupation_groups
    if occ_title.replace(" ", "_").replace("/", "_") in already_complete
)
n_to_process = n_total - n_skip

print(f"Resume status: {n_skip}/{n_total} occupations already complete; {n_to_process} to process.")
print(f"  raw_outputs:              {raw_output_path}")
print(f"  raw_outputs_in_progress:  {in_progress_path}")
print()

# Process the remaining occupations
processed_idx = 0
for i, ((soc_code, occ_title), group) in enumerate(occupation_groups, 1):
    safe_title = occ_title.replace(" ", "_").replace("/", "_")
    if safe_title in already_complete:
        continue
    processed_idx += 1
    tasks_df = group[['Task ID', 'Task Title']].drop_duplicates().sort_values('Task ID').reset_index(drop=True)
    print(f"\n[{processed_idx}/{n_to_process}] (overall {i}/{n_total}) {occ_title}  ({soc_code})")
    score_occupation_handoffs(soc_code, occ_title, tasks_df, raw_output_path, in_progress_path)

print('\n' + '=' * 50)
print('STAGE 1 RUN COMPLETE')
print('=' * 50)
# Recount what's complete now (post-run)
final_complete = sum(
    1 for entry in os.listdir(raw_output_path)
    if os.path.isdir(os.path.join(raw_output_path, entry))
) if os.path.isdir(raw_output_path) else 0
in_progress_count = sum(
    1 for entry in os.listdir(in_progress_path)
    if os.path.isdir(os.path.join(in_progress_path, entry))
) if os.path.isdir(in_progress_path) else 0
print(f'• Fully complete after this run:  {final_complete}/{n_total}')
print(f'• Still in progress (staging):    {in_progress_count}')

# 3) Stage 2 — Concatenate Per-Pair JSONs into a Per-Occupation CSV

For each occupation directory under `raw_outputs/`, read every per-pair JSON file, extract the structured fields, and concatenate into a single CSV under `parsed_outputs/{Occupation_Title}.csv`. Columns:

`O*NET-SOC Code, Occupation Title, Task ID 1, Task Title 1, Task ID 2, Task Title 2, handoff_score, handoff_pct_of_task_time`

This step makes no API calls — re-running it after schema changes is essentially free.

In [26]:
import glob

occ_dirs = sorted([d for d in glob.glob(f"{raw_output_path}/*") if os.path.isdir(d)])
n_occs = 0
n_pairs_total = 0

for occ_dir in occ_dirs:
    safe_title = os.path.basename(occ_dir)
    pair_files = sorted(glob.glob(f"{occ_dir}/*.json"))  # excludes .failed.txt

    rows = []
    for pf in pair_files:
        try:
            with open(pf) as f:
                d = json.load(f)
        except (json.JSONDecodeError, OSError):
            continue
        rows.append({
            'O*NET-SOC Code': d.get('soc_code'),
            'Occupation Title': d.get('occupation_title'),
            'Task ID 1': d.get('task_a', {}).get('id'),
            'Task Title 1': d.get('task_a', {}).get('title'),
            'Task ID 2': d.get('task_b', {}).get('id'),
            'Task Title 2': d.get('task_b', {}).get('title'),
            'handoff_score': d.get('handoff_score'),
            'handoff_pct_of_task_time': d.get('handoff_pct_of_task_time'),
        })

    if not rows:
        print(f"   ⚠️  No parseable pair files in {safe_title}; skipping")
        continue

    df = pd.DataFrame(rows).sort_values(['Task ID 1', 'Task ID 2']).reset_index(drop=True)
    out_path = f"{parsed_output_path}/{safe_title}.csv"
    df.to_csv(out_path, index=False)
    n_occs += 1
    n_pairs_total += len(df)
    print(f"   ✅ {safe_title}: {len(df)} rows → {out_path}")

print(f"\n✓ Wrote {n_occs} per-occupation CSVs to {parsed_output_path}")
print(f"✓ Total rows across all files: {n_pairs_total:,}")

   ✅ Pile_Driver_Operators: 20 rows → ../data/computed_objects/taskHandoffCost_LLM_sample/parsed_outputs/Pile_Driver_Operators.csv
   ✅ Shampooers: 12 rows → ../data/computed_objects/taskHandoffCost_LLM_sample/parsed_outputs/Shampooers.csv

✓ Wrote 2 per-occupation CSVs to ../data/computed_objects/taskHandoffCost_LLM_sample/parsed_outputs
✓ Total rows across all files: 32


In [27]:
# Clean up caffeinate process
try:
    if 'caff_process' in globals() and caff_process is not None:
        caff_process.terminate()
        caff_process.wait()  # Wait for process to terminate
        print("Caffeinate mode OFF 💡 - System sleep is now enabled.")
    else:
        print("Caffeinate was not running or already stopped.")
except Exception as e:
    print(f"Note: {e}")
    print("Caffeinate process may have already ended.")

Caffeinate mode OFF 💡 - System sleep is now enabled.
